In [13]:
from __future__ import annotations

import json
from dataclasses import dataclass
from pathlib import Path
from typing import Dict, List, Set, Tuple

import pandas as pd
from collections import defaultdict, deque


# =========================
# Config
# =========================

@dataclass
class CFG:
    data_dir: Path = Path("data")
    classes_tsv: str = "classes.tsv"
    taxonomy_json: str = "timel_taxonomy.json"


# =========================
# Step 1.1: Load classes
# =========================

def load_classes(classes_path: Path) -> Tuple[List[str], Dict[str, int], List[str]]:
    """
    Returns:
      - labels: list[str] (2333 timel_id like 'tm-xxxx')
      - label2idx
      - idx2label
    """
    df = pd.read_csv(classes_path, sep="\t", dtype=str)

    # ✅ 最小修改：明确使用 timel_id
    if "timel_id" in df.columns:
        labels = df["timel_id"].tolist()
    elif "label" in df.columns:  # 兼容旧格式
        labels = df["label"].tolist()
    else:
        labels = df.iloc[:, 0].tolist()

    labels = [x.strip() for x in labels if isinstance(x, str) and x.strip()]

    if len(labels) != len(set(labels)):
        dup = pd.Series(labels).value_counts()
        dup = dup[dup > 1].index.tolist()[:20]
        raise ValueError(f"Duplicate labels found in classes.tsv, e.g. {dup}")

    label2idx = {lab: i for i, lab in enumerate(labels)}
    idx2label = labels[:]
    return labels, label2idx, idx2label


# =========================
# Step 1.2: Load taxonomy (paths -> DAG)
# =========================

def load_taxonomy_dag_from_paths(
    taxonomy_path: Path
) -> Tuple[Dict[str, Set[str]], Dict[str, Set[str]], Set[str], Dict[str, str]]:
    """
    Parse taxonomy format:
      {"items":[ {"value": "...", "path_ids": ["tm-root", ..., "tm-node"]}, ... ]}

    Returns:
      parents: child -> set(parents)
      children: parent -> set(children)
      nodes: set(all node ids)
      id2value: node_id -> display name (best-effort; only for last node in each path)
    """
    data = json.loads(taxonomy_path.read_text(encoding="utf-8"))
    if not isinstance(data, dict) or "items" not in data or not isinstance(data["items"], list):
        raise ValueError("Expected taxonomy.json to have top-level key 'items' as a list.")

    parents: Dict[str, Set[str]] = defaultdict(set)
    children: Dict[str, Set[str]] = defaultdict(set)
    nodes: Set[str] = set()
    id2value: Dict[str, str] = {}

    for it in data["items"]:
        if not isinstance(it, dict):
            continue

        value = it.get("value")
        path_ids = it.get("path_ids")
        if not isinstance(path_ids, list):
            continue

        path = [x.strip() for x in path_ids if isinstance(x, str) and x.strip()]
        if not path:
            continue

        # "value" corresponds to the *last* id in path
        if isinstance(value, str) and value.strip():
            id2value[path[-1]] = value.strip()

        for a in path:
            nodes.add(a)

        for p, c in zip(path[:-1], path[1:]):
            parents[c].add(p)
            children[p].add(c)

    return dict(parents), dict(children), nodes, id2value


# =========================
# Step 1.3: Clip taxonomy to output space (2333 labels)
# =========================

def clip_taxonomy_to_classes_dag(
    labels: List[str],
    parents: Dict[str, Set[str]],
    children: Dict[str, Set[str]],
    nodes: Set[str],
) -> Dict[str, object]:
    """
    Keep only nodes that exist in classes.tsv (output space).
    """
    label_set = set(labels)

    nodes_in_classes = sorted(list(nodes & label_set))
    nodes_not_in_classes = sorted(list(nodes - label_set))
    classes_not_in_taxonomy = sorted(list(label_set - nodes))

    parents_c: Dict[str, Set[str]] = {}
    for c, ps in parents.items():
        if c in label_set:
            keep = {p for p in ps if p in label_set}
            if keep:
                parents_c[c] = keep

    children_c: Dict[str, Set[str]] = {}
    for p, ch in children.items():
        if p in label_set:
            keep = {c for c in ch if c in label_set}
            if keep:
                children_c[p] = keep

    return {
        "nodes_in_classes": nodes_in_classes,
        "nodes_not_in_classes": nodes_not_in_classes,
        "classes_not_in_taxonomy": classes_not_in_taxonomy,
        "parents_clip": parents_c,
        "children_clip": children_c,
    }


# =========================
# Step 1.4: Ancestors (DAG)
# =========================

def compute_ancestors_dag(
    parents: Dict[str, Set[str]],
    nodes: List[str],
) -> Dict[str, Set[str]]:
    """
    ancestors[n] includes n itself.
    DAG-safe BFS upward.
    """
    ancestors: Dict[str, Set[str]] = {}
    for n in nodes:
        seen = {n}
        q = deque([n])
        while q:
            u = q.popleft()
            for p in parents.get(u, set()):
                if p not in seen:
                    seen.add(p)
                    q.append(p)
        ancestors[n] = seen
    return ancestors


# =========================
# Step 1.5: Depth for DAG (longest path to a root)
# =========================

def compute_depth_dag_longest(
    parents: Dict[str, Set[str]],
    nodes: List[str],
) -> Dict[str, int]:
    """
    For DAG, depth isn't unique. We use longest distance to any root within clipped space.
    Kahn-like DP on edges parent->child.
    """
    node_set = set(nodes)

    # Build children adjacency and indegree inside clipped space
    children = defaultdict(set)
    indeg = {n: 0 for n in nodes}
    for c, ps in parents.items():
        if c not in node_set:
            continue
        for p in ps:
            if p in node_set:
                children[p].add(c)
                indeg[c] += 1

    # Roots: nodes with no parents inside clipped space
    q = deque([n for n in nodes if indeg[n] == 0])
    depth = {n: 0 for n in nodes}

    visited = 0
    while q:
        u = q.popleft()
        visited += 1
        du = depth[u]
        for v in children.get(u, set()):
            if depth[v] < du + 1:
                depth[v] = du + 1
            indeg[v] -= 1
            if indeg[v] == 0:
                q.append(v)

    # If taxonomy has cycles (shouldn't), visited < len(nodes)
    if visited < len(nodes):
        # We don't hard-fail here, but you should investigate if this happens.
        # Cycles would break closure semantics.
        print(f"[WARN] depth: visited {visited}/{len(nodes)} nodes. Potential cycle or disconnected indegree issue.")

    return depth


# =========================
# Step 1.6: (Optional) Descendants (DAG)
# Useful later for consistency filtering / WUP-like logic, but can be heavy.
# We'll keep it optional.
# =========================

def compute_descendants_dag(
    children: Dict[str, Set[str]],
    nodes: List[str],
) -> Dict[str, Set[str]]:
    """
    descendants[n] excludes n itself.
    DAG-safe DFS with memo.
    """
    node_set = set(nodes)
    memo: Dict[str, Set[str]] = {}

    def dfs(u: str) -> Set[str]:
        if u in memo:
            return memo[u]
        out = set()
        for v in children.get(u, set()):
            if v in node_set:
                out.add(v)
                out |= dfs(v)
        memo[u] = out
        return out

    return {n: dfs(n) for n in nodes}


In [14]:
# =========================
# Step 1 Main
# =========================

def step1_build_taxonomy_structures(cfg: CFG, build_descendants: bool = False) -> Dict[str, object]:
    classes_path = cfg.data_dir / cfg.classes_tsv
    taxonomy_path = cfg.data_dir / cfg.taxonomy_json

    labels, label2idx, idx2label = load_classes(classes_path)

    parents, children, nodes, id2value = load_taxonomy_dag_from_paths(taxonomy_path)
    align = clip_taxonomy_to_classes_dag(labels, parents, children, nodes)
    parents_c: Dict[str, Set[str]] = align["parents_clip"]
    children_c: Dict[str, Set[str]] = align["children_clip"]

    ancestors = compute_ancestors_dag(parents_c, labels)
    depth = compute_depth_dag_longest(parents_c, labels)

    descendants = None
    if build_descendants:
        descendants = compute_descendants_dag(children_c, labels)

    report = {
        "n_labels": len(labels),
        "n_tax_nodes_total": len(nodes),
        "n_nodes_in_classes": len(align["nodes_in_classes"]),
        "n_nodes_not_in_classes": len(align["nodes_not_in_classes"]),
        "n_classes_not_in_taxonomy": len(align["classes_not_in_taxonomy"]),
        "n_nodes_with_multi_parents": sum(1 for _, ps in parents_c.items() if len(ps) > 1),
    }

    return {
        "labels": labels,
        "label2idx": label2idx,
        "idx2label": idx2label,
        "parents": parents_c,
        "children": children_c,
        "ancestors": ancestors,
        "depth": depth,
        "descendants": descendants,
        "id2value": id2value,
        "report": report,
    }

In [15]:
# =========================
# Run Step 1
# =========================

if __name__ == "__main__":
    cfg = CFG()
    tax = step1_build_taxonomy_structures(cfg, build_descendants=False)
    print(tax["report"])

    # Coverage sanity check
    label_set = set(tax["labels"])
    node_set = set(tax["id2value"].keys()) | set(tax["parents"].keys()) | set(tax["children"].keys())
    print("labels in taxonomy:", len(label_set & node_set), "/", len(label_set))
    print("labels NOT in taxonomy:", len(label_set - node_set))
    # Multi-parent examples
    multi = [k for k, ps in tax["parents"].items() if len(ps) > 1]
    print("example multi-parent node:", multi[0] if multi else None)

{'n_labels': 2333, 'n_tax_nodes_total': 4576, 'n_nodes_in_classes': 2333, 'n_nodes_not_in_classes': 2243, 'n_classes_not_in_taxonomy': 0, 'n_nodes_with_multi_parents': 0}
labels in taxonomy: 2333 / 2333
labels NOT in taxonomy: 0
example multi-parent node: None


Step2

In [16]:
import ast
import pandas as pd
from pathlib import Path
from typing import Dict, List, Set, Tuple, Optional


def _pick_image_id_col(df: pd.DataFrame) -> str:
    # 常见列名优先
    for c in ["image_id", "id", "img_id", "img", "image", "filename", "file", "path", "img_url"]:
        if c in df.columns:
            return c
    # 否则用第一列
    return df.columns[0]


def _pick_label_source(df: pd.DataFrame) -> Tuple[str, List[str]]:
    """
    Returns:
      mode: "list_col" or "tag_cols"
      cols: [colname] or tag columns
    """
    # tag1-tag9 优先检测
    tag_cols = [c for c in df.columns if c.lower().startswith("tag")]
    if len(tag_cols) >= 2:
        # 按数字排序：tag1, tag2...
        def key(c):
            s = "".join(ch for ch in c if ch.isdigit())
            return int(s) if s else 9999
        tag_cols = sorted(tag_cols, key=key)
        return "tag_cols", tag_cols

    # 否则找可能的“列表列”
    for c in ["mots_cles", "labels", "label_list", "timel_labels", "tags"]:
        if c in df.columns:
            return "list_col", [c]

    # 再否则：尝试找“像列表字符串”的列
    # 取前几行看是否包含 '[' 或 ',' 等
    for c in df.columns:
        sample = df[c].dropna().astype(str).head(50)
        if len(sample) == 0:
            continue
        score = sample.str.contains(r"\[|\]|,").mean()
        if score > 0.4:
            return "list_col", [c]

    raise ValueError("Could not detect label columns in all.tsv (neither tag1.. nor list-col).")


def _parse_list_cell(x: object) -> List[str]:
    """
    Parse a cell that may be:
      - python-list-like string: "['tm-..','tm-..']"
      - comma separated: "tm-a, tm-b"
      - single label
      - NaN
    """
    if x is None or (isinstance(x, float) and pd.isna(x)):
        return []
    if isinstance(x, list):
        return [str(t).strip() for t in x if str(t).strip()]

    s = str(x).strip()
    if not s:
        return []

    # try literal_eval for "['a','b']"
    if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
        try:
            obj = ast.literal_eval(s)
            if isinstance(obj, (list, tuple, set)):
                return [str(t).strip() for t in obj if str(t).strip()]
        except Exception:
            pass

    # fallback: comma-separated
    if "," in s:
        return [t.strip() for t in s.split(",") if t.strip()]

    return [s]

In [17]:
def load_all_labels_map(all_path: Path) -> Tuple[Dict[str, Set[str]], str, Tuple[str, List[str]]]:
    df = pd.read_csv(all_path, sep="\t", dtype=str)
    img_col = _pick_image_id_col(df)
    mode, cols = _pick_label_source(df)

    labels_map: Dict[str, Set[str]] = {}

    if mode == "tag_cols":
        tag_cols = cols
        for _, row in df.iterrows():
            img_id = str(row[img_col]).strip()
            if not img_id:
                continue
            tags = set()
            for c in tag_cols:
                v = row.get(c)
                if v is None or (isinstance(v, float) and pd.isna(v)):
                    continue
                t = str(v).strip()
                if t:
                    tags.add(t)
            labels_map[img_id] = tags

    else:  # list_col
        col = cols[0]
        for _, row in df.iterrows():
            img_id = str(row[img_col]).strip()
            if not img_id:
                continue
            tags = set(_parse_list_cell(row.get(col)))
            labels_map[img_id] = tags

    return labels_map, img_col, (mode, cols)

In [18]:
def load_split_ids(split_path: Path) -> List[str]:
    df = pd.read_csv(split_path, sep="\t", dtype=str)
    img_col = _pick_image_id_col(df)
    ids = df[img_col].dropna().astype(str).str.strip().tolist()
    ids = [x for x in ids if x]
    return ids

In [19]:
import numpy as np


def closure_labels(raw: Set[str], ancestors: Dict[str, Set[str]]) -> Set[str]:
    """
    raw: raw labels of an image
    ancestors: ancestors[label] includes itself
    return: raw union all ancestors
    """
    out = set()
    for lab in raw:
        if lab in ancestors:
            out |= ancestors[lab]
        else:
            # label not found in taxonomy-clip; keep it anyway (still in classes space maybe)
            out.add(lab)
    return out


def multihot_from_labels(
    labs: Set[str],
    label2idx: Dict[str, int],
) -> np.ndarray:
    y = np.zeros((len(label2idx),), dtype=np.float32)
    for lab in labs:
        j = label2idx.get(lab)
        if j is not None:
            y[j] = 1.0
    return y

In [20]:
from dataclasses import dataclass

@dataclass
class Sample:
    image_id: str
    raw_labels: Set[str]
    closed_labels: Set[str]
    y: np.ndarray


def build_split_samples(
    split_ids: List[str],
    all_labels_map: Dict[str, Set[str]],
    ancestors: Dict[str, Set[str]],
    label2idx: Dict[str, int],
) -> Tuple[List[Sample], Dict[str, int]]:
    """
    Returns:
      samples: list[Sample]
      missing: dict counters for diagnostics
    """
    samples: List[Sample] = []
    missing = {"missing_in_all_labels_map": 0, "empty_labels": 0}

    for img_id in split_ids:
        raw = all_labels_map.get(img_id)
        if raw is None:
            missing["missing_in_all_labels_map"] += 1
            raw = set()

        if len(raw) == 0:
            missing["empty_labels"] += 1

        closed = closure_labels(raw, ancestors)
        y = multihot_from_labels(closed, label2idx)

        samples.append(Sample(image_id=img_id, raw_labels=raw, closed_labels=closed, y=y))

    return samples, missing


def compute_train_freq(train_samples: List[Sample]) -> np.ndarray:
    """
    freq_c: count of positives per class across train (closure-applied)
    """
    Y = np.stack([s.y for s in train_samples], axis=0)  # [N, C]
    freq = Y.sum(axis=0)  # [C]
    return freq.astype(np.float32)

In [21]:
def step2_build_datasets(cfg: CFG, tax: Dict[str, object]) -> Dict[str, object]:
    all_path = cfg.data_dir / "all.tsv"
    train_path = cfg.data_dir / "train.tsv"
    dev_path = cfg.data_dir / "dev.tsv"

    all_labels_map, img_col_all, label_src = load_all_labels_map(all_path)
    train_ids = load_split_ids(train_path)
    dev_ids = load_split_ids(dev_path)

    train_samples, train_missing = build_split_samples(
        train_ids,
        all_labels_map,
        tax["ancestors"],
        tax["label2idx"],
    )
    dev_samples, dev_missing = build_split_samples(
        dev_ids,
        all_labels_map,
        tax["ancestors"],
        tax["label2idx"],
    )

    freq = compute_train_freq(train_samples)

    return {
        "train_samples": train_samples,
        "dev_samples": dev_samples,
        "freq": freq,
        "diagnostics": {
            "all_img_col": img_col_all,
            "all_label_source": label_src,  # ("tag_cols", [...]) or ("list_col", [col])
            "n_all": len(all_labels_map),
            "n_train": len(train_samples),
            "n_dev": len(dev_samples),
            "train_missing": train_missing,
            "dev_missing": dev_missing,
            "freq_nonzero": int((freq > 0).sum()),
        },
    }


# run
data_pack = step2_build_datasets(cfg, tax)
data_pack["diagnostics"]

{'all_img_col': 'Image',
 'all_label_source': ('list_col', ['timel_labels']),
 'n_all': 9132,
 'n_train': 7210,
 'n_dev': 1922,
 'train_missing': {'missing_in_all_labels_map': 0, 'empty_labels': 0},
 'dev_missing': {'missing_in_all_labels_map': 0, 'empty_labels': 0},
 'freq_nonzero': 0}

Step3

In [22]:
# ===== Cell 1: define tax (required by Step 3) =====

import pandas as pd

classes_df = pd.read_csv("02_source_data/data/classes.tsv", sep="\t", dtype=str)

if "timel_id" in classes_df.columns:
    labels = classes_df["timel_id"].tolist()
else:
    labels = classes_df.iloc[:, 0].tolist()

tax = {
    "labels": labels
}

assert len(tax["labels"]) == 2333

In [25]:
from pathlib import Path

image_root = Path("odil")

train_ds = OdilMultiLabelDataset(
    data_pack["train_samples"],
    image_root=image_root,
    transform=build_train_transform(384),
)

dev_ds = OdilMultiLabelDataset(
    data_pack["dev_samples"],
    image_root=image_root,
    transform=build_eval_transform(384),
)

NameError: name 'OdilMultiLabelDataset' is not defined

In [23]:
# =========================
# Step 3 — Training (torch-only, vision-only, macOS-safe)
# =========================

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler
from torch.cuda.amp import GradScaler
from transformers import AutoModel, get_cosine_schedule_with_warmup
from tqdm import tqdm


# --------
# 3.1 Vision-only SigLIP2 model
# --------

class SigLIP2MultiLabel(nn.Module):
    def __init__(self, model_name: str, num_labels: int):
        super().__init__()

        backbone = AutoModel.from_pretrained(model_name)

        if hasattr(backbone, "vision_model"):
            self.vision = backbone.vision_model
            cfg = backbone.config.vision_config
        else:
            self.vision = backbone
            cfg = backbone.config

        hidden = cfg.hidden_size
        self.norm = nn.LayerNorm(hidden)
        self.drop = nn.Dropout(0.1)
        self.head = nn.Linear(hidden, num_labels)

    def forward(self, pixel_values):
        out = self.vision(pixel_values=pixel_values)

        if hasattr(out, "pooler_output") and out.pooler_output is not None:
            emb = out.pooler_output
        else:
            emb = out.last_hidden_state[:, 0]

        emb = self.norm(emb)
        emb = self.drop(emb)
        return self.head(emb)


# --------
# 3.2 Asymmetric Loss
# --------

class AsymmetricLoss(nn.Module):
    def __init__(self, gamma_pos=0.0, gamma_neg=4.0, clip=0.05, eps=1e-8):
        super().__init__()
        self.gamma_pos = gamma_pos
        self.gamma_neg = gamma_neg
        self.clip = clip
        self.eps = eps

    def forward(self, logits, targets):
        prob = torch.sigmoid(logits)
        prob = torch.clamp(prob, self.eps, 1 - self.eps)
        prob_neg = torch.clamp(1 - prob + self.clip, max=1.0)

        loss_pos = targets * torch.log(prob)
        loss_neg = (1 - targets) * torch.log(prob_neg)

        if self.gamma_pos > 0:
            loss_pos *= torch.pow(1 - prob, self.gamma_pos)
        if self.gamma_neg > 0:
            loss_neg *= torch.pow(prob, self.gamma_neg)

        return -(loss_pos + loss_neg).sum(dim=1).mean()


# --------
# 3.3 Logit Adjustment (torch-only)
# --------

def build_logit_adjustment_by_images(freq, n_images, tau=1.0, eps=1e-12):
    if not torch.is_tensor(freq):
        freq = torch.tensor(freq, dtype=torch.float32)

    pi = freq / max(n_images, 1)
    pi = torch.clamp(pi, min=eps, max=1.0)

    adj = -tau * torch.log(pi)
    adj[pi <= eps] = 0.0
    return adj


# --------
# 3.4 Tail-aware sampler (no numpy)
# --------

def build_tail_sampler(train_samples, freq, tail_max_count=20, base=1.0, boost=6.0):
    if not torch.is_tensor(freq):
        freq = torch.tensor(freq, dtype=torch.float32)

    tail_mask = (freq > 0) & (freq <= tail_max_count)
    tail_idx = torch.nonzero(tail_mask, as_tuple=False).view(-1).tolist()

    weights = [base] * len(train_samples)
    for i, s in enumerate(train_samples):
        for j in tail_idx:
            if s.y[j] > 0:
                weights[i] *= boost
                break

    return WeightedRandomSampler(
        weights=torch.tensor(weights, dtype=torch.double),
        num_samples=len(weights),
        replacement=True,
    )


# --------
# 3.5 Train / Eval loops
# --------

def train_one_epoch(
    model,
    loader,
    optimizer,
    loss_fn,
    logit_adj,
    device,
    scaler=None,
    scheduler=None,
    epoch=0,
):
    model.train()
    total = 0.0

    for step, batch in enumerate(tqdm(loader, desc=f"train epoch {epoch}", total=len(loader))):
        imgs = batch["image"].to(device)
        y = batch["y"].to(device)

        optimizer.zero_grad(set_to_none=True)

        if scaler is not None:
            with torch.cuda.amp.autocast():
                logits = model(imgs) + logit_adj
                loss = loss_fn(logits, y)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(imgs) + logit_adj
            loss = loss_fn(logits, y)
            loss.backward()
            optimizer.step()

        if scheduler is not None:
            scheduler.step()

        total += loss.item()

    return total / len(loader)


@torch.no_grad()
def eval_logits(model, loader, device):
    model.eval()
    ids, logits = [], []
    for batch in tqdm(loader, desc="eval", total=len(loader)):
        imgs = batch["image"].to(device)
        out = model(imgs)
        logits.append(out.cpu())
        ids.extend(batch["image_id"])
    return ids, torch.cat(logits, dim=0)

/Users/longquanwen0813/miniconda3/envs/hackathon/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [24]:
# =========================
# Run Step 3
# =========================

device = "cuda" if torch.cuda.is_available() else "cpu"

model_name = "google/siglip2-base-patch16-384"
num_labels = len(tax["labels"])

model = SigLIP2MultiLabel(model_name, num_labels).to(device)

loss_fn = AsymmetricLoss(gamma_pos=0.0, gamma_neg=4.0, clip=0.05)

logit_adj = build_logit_adjustment_by_images(
    freq=data_pack["freq"],
    n_images=len(data_pack["train_samples"]),
    tau=1.0,
).to(device)

sampler = build_tail_sampler(
    data_pack["train_samples"],
    data_pack["freq"],
)

train_loader = DataLoader(
    train_ds,
    batch_size=32,
    sampler=sampler,
    num_workers=0,
    pin_memory=False,
)

dev_loader = DataLoader(
    dev_ds,
    batch_size=32,
    shuffle=False,
    num_workers=0,
    pin_memory=False,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5, weight_decay=0.05)

epochs = 5
steps_per_epoch = len(train_loader)
scheduler = get_cosine_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * epochs * steps_per_epoch),
    num_training_steps=epochs * steps_per_epoch,
)

scaler = GradScaler() if device == "cuda" else None

for ep in range(epochs):
    loss = train_one_epoch(
        model,
        train_loader,
        optimizer,
        loss_fn,
        logit_adj,
        device,
        scaler=scaler,
        scheduler=scheduler,
        epoch=ep,
    )
    print(f"epoch {ep} loss={loss:.4f}")

dev_ids, dev_logits = eval_logits(model, dev_loader, device)

torch.save(
    {"dev_ids": dev_ids, "dev_logits": dev_logits, "labels": tax["labels"]},
    "dev_logits.pt",
)
torch.save(
    {"state_dict": model.state_dict(), "labels": tax["labels"]},
    "model_siglip2_multilabel.pt",
)

NameError: name 'train_ds' is not defined